In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv


In [25]:
import pandas as pd
import numpy as np
import re
import os
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Data Loading

In [7]:
DATA_PATH = '/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv'

columns = ["target", "ids", "date", "flag", "user", "text"]
print("Loading dataset from Kaggle input...")
df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1', names=columns)

Loading dataset from Kaggle input...


In [12]:
df.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [11]:
df.text.head()

0    @switchfoot http://twitpic.com/2y1zl - Awww, t...
1    is upset that he can't update his Facebook by ...
2    @Kenichan I dived many times for the ball. Man...
3      my whole body feels itchy and like its on fire 
4    @nationwideclass no, it's not behaving at all....
Name: text, dtype: object

In [13]:
df.target.value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

## Cleaning the Data

In [14]:
print("Cleaning and preparing data...")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'@\w+', '', text) # Remove Mentions
    text = re.sub(r'[^\w\s]', '', text) # Remove Punctuation
    return text.strip()

# Selection: Keep only target and text
df = df[['target', 'text']]

# Label Mapping: 0=Negative, 4=Positive -> map 4 to 1 for binary (0, 1)
df['target'] = df['target'].replace(4, 1)

# Apply cleaning
df['text'] = df['text'].apply(clean_text)
df = df[df['text'] != ""] # Remove any rows that became empty after cleaning

# Sampling 100,000 rows for efficient training on Kaggle GPUs
df = df.sample(n=100000, random_state=42)


Cleaning and preparing data...


## Tokenization and Dataset Creation

In [15]:
print("Preparing Tokenizer and Datasets...")
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Convert to HuggingFace Dataset format
train_ds = Dataset.from_pandas(train_df).rename_column("target", "label")
val_ds = Dataset.from_pandas(val_df).rename_column("target", "label")

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

# Dynamic padding for better performance
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Preparing Tokenizer and Datasets...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## Compute Metrics

In [23]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Model Training

In [20]:
print("Initializing DistilBERT...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # Optimized for Kaggle's NVIDIA GPUs
    report_to="none"
)

Initializing DistilBERT...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, 
)

print("Starting Training Loop...")
trainer.train()

Starting Training Loop...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.427000,0.944552,0.805400,0.797080,0.841665,0.756982
2,0.476252,0.920461,0.824400,0.825030,0.830158,0.819964


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2814, training_loss=0.4883829806828719, metrics={'train_runtime': 470.1814, 'train_samples_per_second': 382.831, 'train_steps_per_second': 5.985, 'total_flos': 1670281429653504.0, 'train_loss': 0.4883829806828719, 'epoch': 2.0})

In [27]:
results = trainer.evaluate()
for key, value in results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss: 0.920461118221283
eval_accuracy: 0.8244
eval_f1: 0.8250298923874053
eval_precision: 0.8301584118708643
eval_recall: 0.8199643493761141
eval_runtime: 7.1097
eval_samples_per_second: 1406.524
eval_steps_per_second: 11.112
epoch: 2.0


In [29]:
SAVE_DIR = "/kaggle/working/social_sentiment_model"
print(f"\nSaving model to {SAVE_DIR}...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil

# Zip the model folder so it can be downloaded manually from the /kaggle/working directory
shutil.make_archive('sentiment_model_zip', 'zip', SAVE_DIR)

print("Success! Download 'sentiment_model_zip.zip' from the Output section.")


Saving model to /kaggle/working/social_sentiment_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Download 'sentiment_model_zip.zip' from the Output section.
